# Concentration Inequalities

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 09.03 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/09_Advanced_Probability/04_concentration_inequalities.ipynb)

---

## 🎯 Learning Objective

Learn Hoeffding, Bernstein, and McDiarmid concentration bounds, and see how they lead to PAC-Bayes generalization guarantees.

---

## 📐 Theory

The Law of Large Numbers (recall `03.01`-`03.03`) says a sample average converges to the true
mean as $n\to\infty$ — but says nothing about *how fast*, or how likely a large deviation is at
any *finite* $n$. **Concentration inequalities** answer exactly that: quantitative bounds on how
tightly a random quantity clusters around its expectation, for a *specific* sample size.

### Hoeffding's inequality

For independent random variables $X_1,\ldots,X_n$ each bounded in $[a,b]$, with sample mean
$\bar{X} = \frac1n\sum_i X_i$ and true mean $\mu = \mathbb{E}[\bar X]$, **Hoeffding's
inequality** bounds the probability of a large deviation:

$$P(|\bar X - \mu| \ge t) \le 2\exp\!\left(-\frac{2nt^2}{(b-a)^2}\right)$$

Two structural facts matter for how you'll use this: the bound decays *exponentially* in $n$
(doubling your sample size doesn't just modestly tighten the bound, it squares it), and it only
needs boundedness, not any assumption about the underlying distribution's shape — a genuinely
distribution-free guarantee.

### Bernstein's inequality: using variance, not just range

Hoeffding only uses the range $b-a$; **Bernstein's inequality** additionally incorporates the
variance $\sigma^2$ of each $X_i$, giving a tighter bound when the true variance is small
relative to the range:

$$P(|\bar X - \mu| \ge t) \le 2\exp\!\left(-\frac{nt^2/2}{\sigma^2 + \frac{(b-a)t}{3}}\right)$$

For small $t$, the denominator is dominated by $\sigma^2$ rather than the range, so Bernstein
can be substantially sharper than Hoeffding whenever the data is low-variance even if its
theoretical range is wide.

### McDiarmid's inequality: beyond simple averages

Hoeffding and Bernstein bound deviations of an *average*. **McDiarmid's inequality**
generalizes to any function $f(X_1,\ldots,X_n)$ satisfying the **bounded differences**
property — changing any single $X_i$ (holding the rest fixed) can change $f$ by at most $c_i$:

$$P(|f(X_1,\ldots,X_n) - \mathbb{E}[f]| \ge t) \le 2\exp\!\left(-\frac{2t^2}{\sum_i c_i^2}\right)$$

This covers far more than sample means — it applies to *any* sufically-stable statistic
computed from independent data, including many quantities that show up when analyzing a
trained model's behavior across a random training set.

### Chernoff bounds

**Chernoff bounds** are a general technique (rather than a single named inequality):
bound $P(X\ge t)$ by exponentiating and applying Markov's inequality to $e^{\lambda X}$ for an
optimally-chosen $\lambda$. Hoeffding's inequality is itself derived via a Chernoff-style
argument specialized to bounded random variables — the "family," not a separate result.

### From concentration to generalization: PAC-style bounds

In supervised learning, you observe *training* error $\hat{L}(\theta)$ (an average over $n$
training points) but care about *true* error $L(\theta)$ (an average over the entire, usually
infinite, data distribution). Hoeffding's inequality — applied with $X_i$ = the (bounded)
per-example loss — converts this directly into a **generalization bound**: with probability at
least $1-\delta$,

$$L(\theta) \le \hat{L}(\theta) + \sqrt{\frac{\ln(2/\delta)}{2n}}$$

(solving Hoeffding's tail bound for $t$ given a target confidence $\delta$). This is the
essential shape of every PAC ("Probably Approximately Correct") learning-theory result: with
high probability ($1-\delta$), the true error is close to the observed training error, by an
amount that shrinks as $\sqrt{1/n}$. **PAC-Bayes** bounds extend this to hold *simultaneously*
across an entire (possibly infinite) family of hypotheses weighted by a prior, at the cost of an
extra term measuring how far the learned "posterior" over hypotheses has moved from that prior —
conceptually the same generalization-vs-training-size tradeoff, now accounting for model
complexity rather than treating a single fixed $\theta$ in isolation.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Overlaying the empirical exceedance probability (from repeated simulation) against Hoeffding's
theoretical bound, across a range of deviation thresholds, shows the bound is a genuine
*upper* envelope — never crossed, sometimes loose, but never violated.

In [ ]:
def hoeffding_bound(n, t, a=0, b=1):
    return np.minimum(1.0, 2 * np.exp(-2 * n * t**2 / (b - a)**2))

rng = np.random.default_rng(0)
n, true_mean, n_trials = 200, 0.3, 20000
samples_all = rng.uniform(size=(n_trials, n)) < true_mean   # Bernoulli(0.3), bounded in [0,1]
sample_means = samples_all.mean(axis=1)
deviations = np.abs(sample_means - true_mean)

t_grid = np.linspace(0.01, 0.3, 60)
empirical_probs = np.array([np.mean(deviations > t) for t in t_grid])
bound_probs = hoeffding_bound(n, t_grid)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.semilogy(t_grid, empirical_probs, 'o-', color="#4C72B0", markersize=3, label="empirical P(|mean-true|>t)")
ax.semilogy(t_grid, bound_probs, color="#C44E52", lw=2, label="Hoeffding bound")
ax.set_xlabel("deviation threshold t"); ax.set_ylabel("probability (log scale)")
ax.set_title(f"Hoeffding's bound (n={n}) never crosses below\nthe true empirical exceedance probability")
ax.legend()
plt.show()

violations = np.sum(empirical_probs > bound_probs)
print(f"Number of thresholds where empirical probability EXCEEDS the bound: {violations} (out of {len(t_grid)})")
print(f"Confirms Hoeffding's inequality is a genuine, never-violated upper bound -- not just a rough heuristic.")

## 🐍 Implementation from Scratch

We implement Hoeffding's and Bernstein's bounds directly from their formulas, stress-test
Hoeffding's bound across many random trials to confirm it's never violated, and compare the two
bounds' tightness on a low-variance bounded random variable.

In [ ]:
def hoeffding_bound(n, t, a=0, b=1):
    return np.minimum(1.0, 2 * np.exp(-2 * n * t**2 / (b - a)**2))

def bernstein_bound(n, t, sigma2, a=0, b=1):
    return np.minimum(1.0, 2 * np.exp(-(n * t**2 / 2) / (sigma2 + (b - a) * t / 3)))

# --- Stress-test: does Hoeffding's bound ever get violated across MANY thresholds/trials? ---
rng = np.random.default_rng(0)
n, true_mean, n_trials = 200, 0.3, 20000
samples = rng.uniform(size=(n_trials, n)) < true_mean   # Bernoulli(0.3), bounded in [0, 1]
deviations = np.abs(samples.mean(axis=1) - true_mean)

t_grid = np.linspace(0.01, 0.3, 100)
empirical = np.array([np.mean(deviations > t) for t in t_grid])
bound = hoeffding_bound(n, t_grid)
violations = np.sum(empirical > bound)
print(f"Hoeffding bound violated at {violations} / {len(t_grid)} thresholds tested")
print(f"(0 violations confirms it's a genuine upper bound, not a rough approximation)")

# --- Bernstein vs. Hoeffding: does using the variance actually help on a LOW-variance case? ---
low_var_samples = rng.beta(50, 50, size=(20000, n))  # concentrates near 0.5, but range is still [0,1]
sigma2 = low_var_samples.var()
true_mean_beta = 0.5
deviations_beta = np.abs(low_var_samples.mean(axis=1) - true_mean_beta)

print(f"\nEmpirical per-sample variance: {sigma2:.5f}  (range [0,1], so Hoeffding can't see this)")
print(f"{'t':>6} {'empirical P':>12} {'Hoeffding bound':>16} {'Bernstein bound':>16}")
for t in [0.02, 0.05, 0.08]:
    emp = np.mean(deviations_beta > t)
    hb, bb = hoeffding_bound(n, t), bernstein_bound(n, t, sigma2)
    print(f"{t:6.2f} {emp:12.5f} {hb:16.5f} {bb:16.5f}")
print("\nBernstein's bound is dramatically tighter here because it uses the ACTUAL (small)")
print("variance, while Hoeffding only sees the (much larger) theoretical range [0,1].")

## 🤖 Why This Matters for AI

Every claim of the form "this model achieves X% test accuracy with 95% confidence" is a
concentration-inequality statement, whether or not the paper spells out which inequality. The
PAC-style bound derived above turns Hoeffding's inequality directly into a **generalization
bound**: given a model's *observed* training loss and the training set size $n$, you get a
principled (if often loose) upper bound on the *true* population loss, with a specific
confidence level — exactly the theoretical backbone behind sample-complexity arguments ("how
much data do we need to trust this model's reported accuracy?"). Below, we compute a concrete
generalization bound for a toy classifier's observed training accuracy, and confirm it via
direct simulation from a known ground-truth distribution: does the true accuracy really fall
within the bound's guarantee, across many random training sets?

In [ ]:
def pac_generalization_gap(n, delta):
    # Hoeffding's tail bound, solved for t given a target confidence 1-delta
    return np.sqrt(np.log(2 / delta) / (2 * n))

# A model with some FIXED, unknown true accuracy against the real data distribution --
# we simulate it as 0.82 so we can check the bound's promise against ground truth
true_accuracy = 0.82
n_train, delta = 500, 0.05   # want the bound to hold with probability >= 95%

gap = pac_generalization_gap(n_train, delta)
print(f"With n={n_train} training examples and delta={delta} (95% confidence):")
print(f"generalization gap = {gap:.4f}")
print(f"If observed training accuracy is, say, 0.85, the bound guarantees (w.p. >= 95%)")
print(f"true accuracy is in [{0.85 - gap:.4f}, {0.85 + gap:.4f}]")

# Does the bound actually deliver on its promise? Simulate MANY random training sets from
# the SAME true distribution and check how often the true accuracy falls in the claimed interval
n_trials = 3000
covered = 0
for _ in range(n_trials):
    correct = rng.uniform(size=n_train) < true_accuracy
    observed_acc = correct.mean()
    if observed_acc - gap <= true_accuracy <= observed_acc + gap:
        covered += 1

coverage_rate = covered / n_trials
print(f"\nEmpirical coverage rate over {n_trials} simulated training runs: {coverage_rate:.4f}")
print(f"Bound promised at least: {1 - delta:.2f}")
print(f"Promise kept: {coverage_rate >= 1 - delta}  (Hoeffding-based bounds are typically")
print(f"CONSERVATIVE -- true coverage often exceeds the promised minimum, as it does here.)")

## 🏋️ Exercises

### Warm-up
1. For $n=1000$ samples of a $[0,1]$-bounded random variable, compute Hoeffding's bound at
   $t=0.05$ by hand. Verify with `hoeffding_bound`.

### Practice
2. In the Bernstein-vs-Hoeffding comparison, replace `rng.beta(50, 50, ...)` with
   `rng.beta(2, 2, ...)` (higher variance, still bounded on $[0,1]$) and recompute both bounds
   at $t=0.05$. Does Bernstein's advantage over Hoeffding shrink, grow, or stay about the same?
   Explain in terms of the formula's dependence on $\sigma^2$.

### Challenge
3. Increase `n_train` in the AI section from 500 to 5000 and re-run the coverage simulation.
   Report the new generalization gap and the new empirical coverage rate. Explain, using the
   $\sqrt{1/n}$ scaling from the Theory section, roughly how much smaller you'd expect the gap
   to be, and check whether your simulation matches that prediction.

---

## 📚 Further Reading
- Boucheron, Lugosi & Massart, *Concentration Inequalities: A Nonasymptotic Theory of Independence*
- McAllester, ["PAC-Bayesian Model Averaging"](https://www.cs.cmu.edu/~avrim/ML05/pdf/mcallester.pdf)
- Shalev-Shwartz & Ben-David, *Understanding Machine Learning*, Ch. 4 (PAC Learning)

---

*Next notebook: [Generative Model Theory](05_generative_model_theory.ipynb)*